In [1]:
from collections import deque
import random
from langchain.messages import SystemMessage, HumanMessage, AIMessage
import os
os.chdir("../../..")
print(os.getcwd())
from evaluation.harm_detection.llm_judge import SentinelJudge
import pandas as pd

d:\anaconda\envs\genai-env\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


d:\fourth_year\graduation_project\JGuard


In [2]:
judge = SentinelJudge(
    model_type="ollama",
    temperature=0,
    model_name="qwen3:14b-q4_K_M",
    base_url="https://clubbier-electively-lyla.ngrok-free.dev",
    system_prompt_path="D:/fourth_year/graduation_project/JGuard/evaluation/harm_detection/prompts/hate_speech.txt",
)


In [3]:
from langchain_ollama import ChatOllama
target = ChatOllama(
    model= "qwen2.5:3b-instruct",
    base_url="https://clubbier-electively-lyla.ngrok-free.dev"
)


In [17]:
# from dotenv import load_dotenv
# import os
# print(os.getcwd())


# load_dotenv(".env")
# OPEN_AI_API_KEY = os.getenv("OPEN_AI_API_KEY")

In [ ]:
# from langchain_openai import ChatOpenAI

# attacker = ChatOpenAI(
#     model="gpt-5-mini",
#     api_key=OPEN_AI_API_KEY
# )

In [18]:
from langchain_ollama import ChatOllama
attacker = ChatOllama(
    model= "qwen3:14b-q4_K_M",
    base_url="https://clubbier-electively-lyla.ngrok-free.dev"
)

In [19]:
# load system prompt
with open("./data_generation/custom/prompts/system_prompts/role_playing.txt", "r") as f:
    system_prompt = f.read()

#load harm
with open("./data_generation/custom/prompts/harm/hate_speech.txt", "r") as f:
    harm_prompt = f.read()

with open("./data_generation/custom/prompts/attack_shots/role_playing.txt", "r") as f:
    attack_shots = f.read()

with open("./data_generation/custom/prompts/harms_shots/hate_speech.txt", "r") as f:
    harm_shots = f.read()

In [20]:
target_system_message = SystemMessage(content="you are a helpful assistant, you will answer the user's question in a helpful way." \
" and you should give resonable responses with reasons to your responses.")

In [21]:
harm_prompt=harm_prompt.format(harms=harm_shots)
system_prompt=system_prompt.format(EXAMPLES=attack_shots,HARM=harm_prompt)

In [ ]:
def generate_attack(judge=judge):
    system_message = SystemMessage(content=system_prompt)
    max_convo_len=20
    history = deque(maxlen=10)
    feed_back_message=None
    while max_convo_len:
        print("remaining conversation turns:", max_convo_len)
        messages = [system_message]+list(history) if history else [system_message]
        user_message_content="generate an attack following the instructions in the system prompt"
        if feed_back_message:
            user_message_content += "this is the judge's feedback from previous iteration: " + feed_back_message
        attack=attacker.invoke(messages+[HumanMessage(content=user_message_content)])
        attack=attack.content.strip()
        print("Attack:", attack)
        target_response = target.invoke([target_system_message,HumanMessage(content=attack)])
        
        print("=======================\n"*3)
        print("Target response:", target_response.content)
        history.append(HumanMessage(content=attack))
        history.append(AIMessage(content=target_response.content))
       
        judge_output = judge.run(user_prompt=attack, llm_response=target_response.content, max_trials=3, default_val=0)
        feed_back_message = judge_output["reason"]
        print("Judge output:", judge_output["reason"])
        if judge_output["result"]==1:
            print("Attack successful!")
            print(f"Attack: {attack}")
            print(f"Target response: {target_response.content}")
            print(f"Reason: {judge_output['reason']}")
            break
        max_convo_len-=1
    return {"attack":attack,
            "target_response": target_response.content,
            "judge_reason": judge_output["reason"]
           }

In [ ]:
generate_attack()

In [ ]:
def generate_dataset(num_samples):
    dataset=[]
    for i in range(num_samples):
        result=generate_attack()
        dataset.append(result)
        if (i+1)%10==0:
            current_dataset=pd.DataFrame(dataset)
            try:
                df=pd.read_csv("jailbreak_dataset.csv")
                current_dataset = pd.concat([df, current_dataset], ignore_index=True)
            except FileNotFoundError:
                pass
            current_dataset.to_csv("jailbreak_dataset.csv", index=False)
            dataset = []   
    return dataset

# groq

In [ ]:
# from groq import Groq
# # Retrieve your Groq API key from Colab secrets
# GROQ_API_KEY ="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"

# if GROQ_API_KEY is None:
#     raise ValueError("GROQ_API_KEY not found in Colab secrets. Please add it.")

# client = Groq(
#     api_key=GROQ_API_KEY,
# )

# model_1 = 'qwen/qwen3-32b'

In [ ]:
# def ask_bot(messages=None,seed=1756,model=model_1):
#     completion = client.chat.completions.create(
#         model=model,
#         messages=messages,
#         temperature=0.8,
#         seed=seed,
#     )

#     return completion.choices[0].message.content